<a href="https://colab.research.google.com/github/ChrisCopeland123/Document_text_Analyzer/blob/main/Project_2_Milestone_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Manning Live Project
### Project 2: Smart Document Q&A System¶
### 3.0 Question_Answering System with RAG

#### Environment setup

In [1]:
import os
import json
import numpy as np
import openai
import re
import pickle
from typing import List, Dict, Any, Tuple
from google.colab import userdata

### 3.1 Answering System with RAG

In [2]:
class DocumentProcessor:
    def __init__(self, model: str = "gpt-3.5-turbo"):
        """Initialize the DocumentProcessor with complete RAG capabilities."""
        self.model = model
        self.documents: Dict[str, str] = {}
        self.vector_store: List[Tuple[str, int, List[float], str]] = []

        # Initialize OpenAI client
        self.client = openai.OpenAI(api_key=userdata.get('OpenAI'))

    # Include all methods from Milestone 2 (chunk_text, _get_embedding, etc.)
    def chunk_text(self, text: str, chunk_size: int = 1000, overlap: int = 200) -> List[str]:
        """Split text into overlapping chunks (from Milestone 1)."""
        if not text:
            return []

        paragraphs = re.split(r'\n\s*\n', text)
        chunks = []
        current_chunk = ""

        for para in paragraphs:
            if len(current_chunk) + len(para) > chunk_size and current_chunk:
                chunks.append(current_chunk.strip())
                current_chunk = current_chunk[-overlap:] if overlap > 0 else ""
            current_chunk += para + "\n\n"

        if current_chunk.strip():
            chunks.append(current_chunk.strip())

        return chunks

    def _get_embedding(self, text: str) -> list:
        """Generate embedding for text using OpenAI's API (from Milestone 1)."""
        if not text:
            return []

        try:
            response = self.client.embeddings.create(
                model="text-embedding-ada-002",
                input=text
            )
            return response.data[0].embedding
        except Exception as e:
            print(f"Error generating embedding: {e}")
            return []

    def _calculate_similarity(self, embedding1: List[float], embedding2: List[float]) -> float:
        """Calculate cosine similarity between two embeddings (from Milestone 2)."""
        if not embedding1 or not embedding2:
            return 0.0

        vec1 = np.array(embedding1)
        vec2 = np.array(embedding2)

        dot_product = np.dot(vec1, vec2)
        norm1 = np.linalg.norm(vec1)
        norm2 = np.linalg.norm(vec2)

        if norm1 == 0 or norm2 == 0:
            return 0.0

        return float(dot_product / (norm1 * norm2))

    def search_similar_chunks(self, query: str, top_k: int = 3) -> List[Dict[str, Any]]:
        """Search for the most similar chunks to the query (from Milestone 2)."""
        if not self.vector_store:
            return []

        query_embedding = self._get_embedding(query)
        if not query_embedding:
            return []

        results = []
        for doc_id, chunk_id, chunk_embedding, chunk_text in self.vector_store:
            similarity = self._calculate_similarity(query_embedding, chunk_embedding)
            result_dict = {
                'doc_id': doc_id,
                'chunk_id': chunk_id,
                'text': chunk_text,
                'similarity': similarity
            }
            results.append(result_dict)

        sorted_results = sorted(results, key=lambda x: x['similarity'], reverse=True)
        return sorted_results[:top_k]

    def answer_question(self, question: str, top_k: int = 3) -> Dict[str, Any]:
        """Answer a question based on the document content using RAG."""

        # Handle empty or invalid questions
        if not question or not question.strip():
            return {
                'answer': "Please provide a valid question.",
                'chunks_used': [],
                'confidence': 0.0,
                'sources': []
            }

        relevant_chunks = self.search_similar_chunks(question, top_k)

        # Check confidence threshold based on similarity scores
        # If max similarity is below 0.5, consider returning "insufficient information" message
        max_similarity = max([chunk['similarity'] for chunk in relevant_chunks])

        if max_similarity < 0.5:
            return {
                'answer': "I do not have sufficient information to answer this question",
                'chunks_used': relevant_chunks,
                'confidence': max_similarity,
                'sources': list(set([chunk['doc_id'] for chunk in relevant_chunks]))
            }

        # Combine relevant chunks into context
        context = "\n\n---\n\n".join([chunk['text'] for chunk in relevant_chunks])

        # Create a prompt for question answering
        prompt = f"""
        Answer the following question based ONLY on the provided context. If the context doesn't contain
        enough information to answer the question completely, say so clearly. Do not use any knowledge
        outside of the provided context.

        Context:
        {context}

        Question: {question}
        """

        try:
            # Make API call to OpenAI for answer generation
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[
                    {"role": "system", "content": "You are a helpful assistant that accurately answers questions based only on the provided context."},
                {"role": "user", "content": prompt}
                ],
                temperature=0.3  # Slightly creative but focused
            )

            # Extract answer from response
            answer = response.choices[0].message.content # Get content from response

            # Return structured response with answer and source information
            return {
                'answer': answer,
                'chunks_used': relevant_chunks,
                'confidence': max_similarity,
                'sources': list(set([chunk['doc_id'] for chunk in relevant_chunks]))
            }

        except Exception as e:
            print(f"Error in answering question: {e}")
            return {
                'answer': f"An error occurred while processing your question: {str(e)}",
                'chunks_used': [],
                'confidence': 0.0,
                'sources': []
            }

    def add_document(self, doc_id: str, text: str) -> bool:
        """Process and add a document to the system (from Milestone 2)."""
        if not text:
            print(f"Error: Empty text for document {doc_id}")
            return False

        self.documents[doc_id] = text
        chunks = self.chunk_text(text)
        print(f"Document {doc_id} split into {len(chunks)} chunks")

        successful_chunks = 0
        for i, chunk in enumerate(chunks):
            embedding = self._get_embedding(chunk)
            if embedding:
                self.vector_store.append((doc_id, i, embedding, chunk))
                successful_chunks += 1

        print(f"Successfully processed {successful_chunks}/{len(chunks)} chunks")
        return successful_chunks > 0

    def add_document_from_file(self, file_path: str) -> bool:
        """Read and add a document from a file."""
        text = read_text_file(file_path)
        if text:
            doc_id = os.path.basename(file_path)
            return self.add_document(doc_id, text)
        return False

    def save_state(self, file_path: str) -> bool:
        """Save the current state of the document processor."""
        try:
            state = {
                'documents': self.documents,
                'vector_store': self.vector_store
            }

            with open(file_path, 'wb') as f:
                pickle.dump(state, f)

            print(f"State saved to {file_path}")
            return True

        except Exception as e:
            print(f"Error saving state: {e}")
            return False

    def load_state(self, file_path: str) -> bool:
        """Load a saved state."""
        try:
            with open(file_path, 'rb') as f:
                state = pickle.load(f)

            self.documents = state['documents']
            self.vector_store = state['vector_store']

            print(f"State loaded from {file_path}")
            print(f"Loaded {len(self.documents)} documents and {len(self.vector_store)} chunks")
            return True

        except Exception as e:
            print(f"Error loading state: {e}")
            return False

    def display_document_stats(self):
        """Display statistics about the documents in the system."""
        if not self.documents:
            print("No documents have been added to the system.")
            return

        print("\n" + "="*60)
        print("DOCUMENT STATISTICS")
        print("="*60)

        print(f"Total documents: {len(self.documents)}")
        print(f"Total chunks: {len(self.vector_store)}")

        chunks_by_doc = {}
        for doc_id, chunk_id, _, _ in self.vector_store:
            if doc_id not in chunks_by_doc:
                chunks_by_doc[doc_id] = 0
            chunks_by_doc[doc_id] += 1

        print("\nChunks per document:")
        for doc_id, count in chunks_by_doc.items():
            char_count = len(self.documents.get(doc_id, ""))
            print(f"  - {doc_id}: {count} chunks ({char_count} characters)")

        print("="*60)

In [3]:
# Interactive helper function
def ask_question(question: str, processor: DocumentProcessor = None, show_sources: bool = True):
    """Ask a question to the DocumentProcessor and display the answer with sources."""

    if processor is None:
        print("Error: No DocumentProcessor provided")
        return

    # Get answer using the answer_question method
    result = processor.answer_question(question)

    # Display the question and answer
    print(f"Question: {question}")
    print("-" * 60)
    print("Answer:")
    print(result['answer'])
    print("-" * 60)

    # Show sources if requested and available
    if show_sources and result.get('chunks_used'):
        print("Source chunks:")
        for i, chunk in enumerate(result['chunks_used']):
            print(f"\nChunk {i+1} (Similarity: {chunk['similarity']:.4f}):")
            print(f"From document: {chunk['doc_id']}")
            print("-" * 40)
            # Display chunk text (truncate if too long). Show first 200 chars with "..." if longer
            chunk_text = chunk["text"]
            if len(chunk_text) > 200:
                text_preview = chunk_text[:200] + "..."
            else:
                text_preview = chunk_text

            print(text_preview)
            print("-" * 40)

    # TODO: Display confidence information if available
    if 'confidence' in result:
        print(f"\nConfidence score: {result['confidence']:.4f}")

    # Display unique sources
    if result.get('sources'):
        print(f"Sources consulted: {', '.join(result['sources'])}")

    return result

In [4]:
# Helper function for reading files
def read_text_file(file_path: str) -> str:
    """Read content from a text file."""
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            return file.read()
    except Exception as e:
        print(f"Error reading file {file_path}: {e}")
        return ""

In [5]:
# Comprehensive test function
def test_question_answering():
    """Test the complete Q&A system with various question types."""

    # Create comprehensive sample documents
    sample_docs = {
        "business_contract.txt": """
        Software Development Agreement

        Parties: TechCorp Inc. (Client) and DevSolutions LLC (Contractor)
        Contract Value: $750,000
        Project Duration: 18 months (January 1, 2024 - June 30, 2025)
        Payment Terms: 40% upfront, 40% at midpoint, 20% upon completion

        Scope of Work:
        - Custom software application development
        - Database design and implementation
        - User interface and experience design
        - Testing and quality assurance
        - Documentation and training materials

        Deliverables:
        - Functional software application
        - Source code and documentation
        - User training sessions (40 hours total)
        - 6-month maintenance and support period

        Liability Limitations:
        - Total liability capped at contract value
        - Excludes damages from force majeure events
        - Both parties maintain mutual confidentiality
        """,

        "performance_report.txt": """
        Q4 2023 Performance Analysis Report

        Executive Summary:
        Overall company performance exceeded targets by 12% in Q4 2023.
        Revenue increased 25% compared to Q4 2022, reaching $2.1 million.
        Customer satisfaction scores averaged 4.7/5.0 across all service lines.

        Key Metrics:
        - Project completion rate: 94% (target: 90%)
        - Client retention rate: 89% (industry average: 75%)
        - Employee satisfaction: 4.3/5.0 (up from 4.1 in Q3)
        - Cost efficiency improved by 8% through process optimization

        Notable Achievements:
        - Successfully delivered 3 major client projects ahead of schedule
        - Implemented new project management system reducing overhead by 15%
        - Expanded team by 5 senior consultants in key growth areas

        Areas for Improvement:
        - Communication response time (target: <24 hours, actual: 28 hours)
        - Documentation completion rate (target: 100%, actual: 87%)
        - Training program participation (target: 95%, actual: 82%)
        """,

        "meeting_minutes.txt": """
        Strategic Planning Meeting - January 15, 2024

        Attendees:
        - Sarah Johnson (CEO)
        - Michael Chen (CTO)
        - Lisa Rodriguez (VP Operations)
        - David Kim (Head of Business Development)

        Agenda Items:

        1. 2024 Growth Strategy
        - Target: 30% revenue increase to $2.8M by end of 2024
        - Plan to hire 8 additional staff members by Q3
        - Focus on AI and automation service offerings
        - Budget approved: $400,000 for technology infrastructure

        2. Client Acquisition Goals
        - Target: 15 new enterprise clients in 2024
        - Expand into healthcare and financial services sectors
        - Marketing budget increased to $150,000
        - Trade show participation: 4 major industry events

        3. Operational Improvements
        - Implement new document management system by March 2024
        - Establish remote work policies for 50% hybrid model
        - Quarterly all-hands meetings starting Q2
        - Performance review cycle updated to semi-annual

        Action Items:
        - Sarah: Finalize hiring plan by January 30
        - Michael: Technology infrastructure proposal by February 15
        - Lisa: Document management vendor selection by February 1
        - David: Q1 marketing campaign launch by February 10
        """
    }

    # Save sample documents
    for filename, content in sample_docs.items():
        with open(filename, 'w', encoding='utf-8') as f:
            f.write(content)

    print("Setting up Q&A System Test...")
    print("=" * 60)

    # Create DocumentProcessor and add documents
    processor = DocumentProcessor()

    # Add all documents to the processor
    for filename in sample_docs.keys():
        success = processor.add_document_from_file(filename)
        print(f"Added {filename}: {'Success' if success else 'Failed'}")

    # Display system statistics
    print("\n2. System Statistics")
    processor.display_document_stats()

    # Test various question types
    test_questions = [
        "What is the contract value?",  # Should find specific factual info
        "Who attended the strategic meeting?",  # Should find attendee list
        "What were the Q4 performance results?",  # Should find performance data
        "When is the project completion deadline?",  # Should find timeline info
        "What is quantum computing?",  # Should indicate no relevant information
        "How much budget was approved for technology?",  # Should find budget info
        ""  # Should handle empty question
    ]

    print("\n3. Testing Question-Answering System:")
    print("=" * 60)

    # Test each question using ask_question helper
    for i, question in enumerate(test_questions, 1):
        print(f"\nTest {i}:")
        if question: # Skip empty question for formatted display
            result = ask_question(question, processor, show_sources=True)
        else:
            print("Testing empty question...")
            result = processor.answer_question(question)
            print(f"Result: {result['answer']}")

        print("\n" + "="*60)



In [6]:
# Advanced testing function for answer quality
def test_answer_quality():
    """Test answer quality with edge cases and confidence scoring."""

    processor = DocumentProcessor()

    # Add a simple test document
    test_doc = """
    Company Policy Document

    Vacation Policy:
    - Employees receive 15 days of paid vacation annually
    - Vacation requests must be submitted 2 weeks in advance
    - Maximum of 5 days can be carried over to the next year

    Remote Work Policy:
    - Employees may work remotely up to 3 days per week
    - Remote work requires manager approval
    - All remote workers must be available during core hours (9 AM - 3 PM)
    """

    processor.add_document("policy.txt", test_doc)

    print("Testing Answer Quality and Edge Cases:")
    print("-" * 50)

    # Test different types of questions
    quality_tests = [
        ("How many vacation days do employees get?", "Should find exact answer"),
        ("What are the remote work requirements?", "Should find multiple requirements"),
        ("What is the salary range?", "Should indicate no information"),
        ("", "Should handle empty question"),
        ("vacation policy requirements", "Should find relevant info with different phrasing")
    ]

    for question, expected in quality_tests:
        print(f"\nQuestion: {question}")
        print(f"Expected: {expected}")
        result = processor.answer_question(question)
        print(f"Answer: {result['answer']}")
        print(f"Confidence: {result['confidence']:.4f}")
        print(f"Sources: {result['sources']}")
        print("-" * 30)

In [7]:
# Run the test
if __name__ == "__main__":
    test_question_answering()
    print("\n" + "=" * 80)
    test_answer_quality()


Setting up Q&A System Test...
Document business_contract.txt split into 1 chunks
Successfully processed 1/1 chunks
Added business_contract.txt: Success
Document performance_report.txt split into 2 chunks
Successfully processed 2/2 chunks
Added performance_report.txt: Success
Document meeting_minutes.txt split into 2 chunks
Successfully processed 2/2 chunks
Added meeting_minutes.txt: Success

2. System Statistics

DOCUMENT STATISTICS
Total documents: 3
Total chunks: 5

Chunks per document:
  - business_contract.txt: 1 chunks (947 characters)
  - performance_report.txt: 2 chunks (1069 characters)
  - meeting_minutes.txt: 2 chunks (1337 characters)

3. Testing Question-Answering System:

Test 1:
Question: What is the contract value?
------------------------------------------------------------
Answer:
The contract value is $750,000.
------------------------------------------------------------
Source chunks:

Chunk 1 (Similarity: 0.7928):
From document: business_contract.txt
---------------